In [ ]:
import os
import pickle
import prince
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.font_manager import FontProperties
from scipy.stats import chi2_contingency

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
version3_path = os.path.join(parent_dir, "Version3")

os.chdir(version3_path)

from sklearn.cluster import AgglomerativeClustering
from tdamapper.core_old import MapperAlgorithm
from tdamapper.cover import CubicalCover
from tdamapper.clustering import FailSafeClustering

from utils.utils_v3 import *
from utils.plots import *
from utils.preprocess import preprocess, process_other, get_unique_ids

try:
    myfont = FontProperties(fname=r"/System/Library/Fonts/PingFang.ttc")
    sns.set(style="whitegrid", font=myfont.get_name())
except Exception as e:
    print(e)

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

dataA2 = pd.read_csv("./Data/A2.csv", low_memory=False)
dataA1 = pd.read_csv("./Data/A1.csv")

In [ ]:
select_lst = [
    # 月份是為了篩選每個月2萬筆
    '發生月份',

    '天候名稱', '光線名稱', 
    '道路類別-第1當事者-名稱', '速限-第1當事者', 
    '路面狀況-路面鋪裝名稱', '路面狀況-路面狀態名稱', '路面狀況-路面缺陷名稱',
    '道路障礙-障礙物名稱', '道路障礙-視距品質名稱', '道路障礙-視距名稱',
    '號誌-號誌種類名稱', '號誌-號誌動作名稱',
    '車道劃分設施-分道設施-快車道或一般車道間名稱', '車道劃分設施-分道設施-快慢車道間名稱', '車道劃分設施-分道設施-路面邊線名稱',
    '當事者屬-性-別名稱', '當事者事故發生時年齡',
    '保護裝備名稱', '行動電話或電腦或其他相類功能裝置名稱',
    '肇事逃逸類別名稱-是否肇逃',
    '死亡受傷人數',

    # 大類別
    '道路型態大類別名稱', '事故位置大類別名稱',
    '車道劃分設施-分向設施大類別名稱',
    '事故類型及型態大類別名稱', '當事者區分-類別-大類別名稱-車種', '當事者行動狀態大類別名稱',
    '車輛撞擊部位大類別名稱-最初', '車輛撞擊部位大類別名稱-其他',

    # 兩個欄位只有兩個觀察值不同
    '肇因研判大類別名稱-主要',
    # '肇因研判大類別名稱-個別',
    
    # 子類別
    '道路型態子類別名稱', '事故位置子類別名稱', '事故類型及型態子類別名稱', '肇因研判子類別名稱-主要',
    '當事者區分-類別-子類別名稱-車種', '當事者行動狀態子類別名稱', '車輛撞擊部位子類別名稱-最初',
    '車輛撞擊部位子類別名稱-其他', '肇因研判子類別名稱-個別',
]

In [ ]:
full_dataA1 = preprocess(dataA1, target='全部', lst=select_lst)
full_dataA2 = preprocess(dataA2, target='全部', lst=select_lst)
mapper_numpy, rbind_data, dummy_data, death, injuried = process_other(full_dataA1, full_dataA2, downsample=False, en=False)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import umap
import prince
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, recall_score
from imblearn.combine import SMOTEENN

X = dummy_data
y = death

In [ ]:
import os
import pickle
import time
import gc

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np

from imblearn.combine import SMOTEENN
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import EditedNearestNeighbours, RandomUnderSampler
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

def logistic_cm_gridsearch(X, y, random_state=42, n_jobs=12):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=random_state)

    smote = SMOTE(random_state=random_state, k_neighbors=3)
    enn = EditedNearestNeighbours(n_neighbors=3)
    smote_enn = SMOTEENN(smote=smote, enn=enn, random_state=random_state)
    X_resampled_train, y_resampled_train = smote_enn.fit_resample(X_train, y_train)

    min_class_count = min(sum(y_test == 0), sum(y_test == 1))
    rus_test = RandomUnderSampler(sampling_strategy={0: min_class_count, 1: min_class_count}, random_state=random_state)
    X_resampled_test, y_resampled_test = rus_test.fit_resample(X_test, y_test)

    model = LogisticRegression(solver='liblinear', max_iter=1000, random_state=random_state)
    parameters = {
        'penalty': ['l2'], 
        'C': [0.01, 1],
    }
    grid_search = GridSearchCV(model, parameters, cv=5, scoring='accuracy', n_jobs=n_jobs)
    grid_search.fit(X_resampled_train, y_resampled_train)
    best_model = grid_search.best_estimator_

    print("Best parameters found by GridSearchCV:", grid_search.best_params_)

    y_proba = best_model.predict_proba(X_resampled_test)[:, 1]

    return y_resampled_test, y_proba, np.arange(len(y_resampled_test))

def linear_svc_cm_gridsearch(X, y, random_state=42, n_jobs=12):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=random_state)

    smote = SMOTE(random_state=random_state, k_neighbors=3)
    enn = EditedNearestNeighbours(n_neighbors=3)
    smote_enn = SMOTEENN(smote=smote, enn=enn, random_state=random_state)
    X_resampled_train, y_resampled_train = smote_enn.fit_resample(X_train, y_train)

    min_class_count = min(sum(y_test == 0), sum(y_test == 1))
    rus_test = RandomUnderSampler(sampling_strategy={0: min_class_count, 1: min_class_count}, random_state=random_state)
    X_resampled_test, y_resampled_test = rus_test.fit_resample(X_test, y_test)

    model = LinearSVC(random_state=random_state, max_iter=500000)

    parameters = {
        'C': [0.01, 0.1, 1, 10, 100],
        'loss': ['hinge', 'squared_hinge']
    }

    grid_search = GridSearchCV(model, parameters, cv=5, scoring='accuracy', n_jobs=n_jobs)
    grid_search.fit(X_resampled_train, y_resampled_train)
    best_model = grid_search.best_estimator_

    print("Best parameters found by GridSearchCV:", grid_search.best_params_)

    decision_scores = best_model.decision_function(X_resampled_test)

    return y_resampled_test, decision_scores, np.arange(len(y_resampled_test))

def xgboost_cm_gridsearch(X, y, random_state=42, n_jobs=12):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=random_state)

    smote = SMOTE(random_state=random_state, k_neighbors=3)
    enn = EditedNearestNeighbours(n_neighbors=3)
    smote_enn = SMOTEENN(smote=smote, enn=enn, random_state=random_state)
    X_resampled_train, y_resampled_train = smote_enn.fit_resample(X_train, y_train)

    min_class_count = min(sum(y_test == 0), sum(y_test == 1))
    rus_test = RandomUnderSampler(sampling_strategy={0: min_class_count, 1: min_class_count}, random_state=random_state)
    X_resampled_test, y_resampled_test = rus_test.fit_resample(X_test, y_test)

    model = XGBClassifier(eval_metric='logloss', random_state=random_state)
    parameters = {
        'n_estimators': [50, 100, 200], # 樹的数量
        'max_depth': [3, 5, 9],
        'learning_rate': [0.01, 0.2],
        'colsample_bytree': [0.8, 1.0],  # 每棵樹使用的特征采样比例
    }
    grid_search = GridSearchCV(model, parameters, cv=5, scoring='accuracy', n_jobs=n_jobs)
    grid_search.fit(X_resampled_train, y_resampled_train)
    best_model = grid_search.best_estimator_

    print("Best parameters found by GridSearchCV:", grid_search.best_params_)

    y_proba = best_model.predict_proba(X_resampled_test)[:, 1]

    return y_resampled_test, y_proba, np.arange(len(y_resampled_test))


In [ ]:
from sklearn.preprocessing import MinMaxScaler

y_binary = np.where(death > 0, 1, 0) 
scaler = MinMaxScaler() 

mca = prince.MCA(n_components=10)
X_mca_raw = mca.fit_transform(rbind_data).to_numpy() 
X_mca = scaler.fit_transform(X_mca_raw)

pca = PCA(n_components=10)
X_pca_raw = pca.fit_transform(dummy_data)
X_pca = scaler.fit_transform(X_pca_raw) 

print("Running UMAP...")
reducer = umap.UMAP(n_components=10)
X_umap_raw = reducer.fit_transform(dummy_data)
X_umap = scaler.fit_transform(X_umap_raw)

In [ ]:
models = [
    ("mca_only", X_mca, y_binary),
    ("pca", X_pca, y_binary),
    ("umap", X_umap, y_binary)
]

for seed in range(40, 41):

    # for name, X, y in models:
    #     print(f'{name} xgboost start')
    #     start_time = time.time()
    #     # Ensure X is float to prevent data type errors
    #     y_xgb, decision_scores_xgb, indices_xgb = xgboost_cm_gridsearch(X.astype(float), y, random_state=seed)
    #     end_time = time.time()
    #     elapsed_time = end_time - start_time
        
    #     save_dir = f"../CompareOther/xgboost"
    #     os.makedirs(save_dir, exist_ok=True)
        
    #     with open(f"{save_dir}/{name}_xgboost.pkl", "wb") as f:
    #         pickle.dump({
    #             'y': y_xgb,
    #             'decision_scores': decision_scores_xgb,
    #             'indices': indices_xgb,
    #             'elapsed_time': elapsed_time
    #         }, f)
    #     print(f'{name} xgboost done in {elapsed_time:.2f} seconds')

    # for name, X, y in models:
    #     print(f'{name} svc start')
    #     start_time = time.time()
    #     y_svc, decision_scores_svc, indices_svc = linear_svc_cm_gridsearch(X.astype(float), y, random_state=seed)
    #     end_time = time.time()
    #     elapsed_time = end_time - start_time

    #     save_dir = f"../CompareOther/svc"
    #     os.makedirs(save_dir, exist_ok=True)

    #     with open(f"{save_dir}/{name}_svc.pkl", "wb") as f:
    #         pickle.dump({
    #             'y': y_svc,
    #             'decision_scores': decision_scores_svc,
    #             'indices': indices_svc,
    #             'elapsed_time': elapsed_time
    #         }, f)
    #     print(f'{name} svc done in {elapsed_time:.2f} seconds')

    for name, X, y in models:
        print(f'{name} logistic start')
        start_time = time.time()
        y_log, decision_scores_log, indices_log = logistic_cm_gridsearch(X.astype(float), y, random_state=seed)
        end_time = time.time()
        elapsed_time = end_time - start_time

        save_dir = f"../CompareOther/logistic"
        os.makedirs(save_dir, exist_ok=True)

        with open(f"{save_dir}/{name}_logistic.pkl", "wb") as f:
            pickle.dump({
                'y': y_log,
                'decision_scores': decision_scores_log,
                'indices': indices_log,
                'elapsed_time': elapsed_time
            }, f)
        print(f'{name} logistic done in {elapsed_time:.2f} seconds')

In [ ]:
def get_optimal_threshold(y_true, decision_scores):
    fpr, tpr, thresholds = roc_curve(y_true, decision_scores)
    youden_j = tpr - fpr
    optimal_idx = np.argmax(youden_j)
    optimal_threshold = thresholds[optimal_idx]
    return optimal_threshold, fpr, tpr, thresholds

def get_score(y_true, decision_scores, threshold=None):
    if threshold is None:
        threshold, fpr, tpr, thresholds = get_optimal_threshold(y_true, decision_scores)
        # plot_roc_curve(fpr, tpr, thresholds, optimal_threshold=threshold)
    
    y_pred = (decision_scores >= threshold).astype(int)
    
    conf_matrix = confusion_matrix(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)
    classification = classification_report(y_true, y_pred, digits=4)
    
    return conf_matrix, recall, precision, f1, accuracy, classification, threshold

def plot_roc_curve(fpr, tpr, thresholds, optimal_threshold):
    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % auc(fpr, tpr))
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.scatter(fpr[np.argmax(tpr - fpr)], tpr[np.argmax(tpr - fpr)], marker='o', color='red', label='Optimal Threshold')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic')
    plt.legend(loc="lower right")
    plt.show()
    
def calculate_recall_precision(confusion_matrix):
    """
    Calculate recall and precision from a given confusion matrix.
    
    Parameters:
        confusion_matrix (numpy.ndarray): A 2x2 confusion matrix.
                                         [[TP, FP],
                                          [FN, TN]]
    
    Returns:
        recall (float): Recall value.
        precision (float): Precision value.
    """
    # Extract values from the confusion matrix
    TP = confusion_matrix[1, 1]
    FP = confusion_matrix[0, 1]
    FN = confusion_matrix[1, 0]
    TN = confusion_matrix[0, 0]

    recall = TP / (TP + FN)
    precision = TP / (TP + FP)
    f1_score = (2 * precision * recall) / (precision + recall)
    acc = (TP + TN) / (TP + FP + FN + TN)

    return recall, precision, f1_score, acc

In [ ]:
ml_models = ["logistic", "svc", "xgboost"]
algorithms = ["mca_only", "pca", "umap"]

algo_display_names = {"mca_only": "MCA", "pca": "PCA", "umap": "UMAP"}
model_display_names = {"logistic": "Logistic", "svc": "SVC", "xgboost": "XGBoost"}

topological_subgroups = [
    'pass_out_overlap', 'pass_0', 'pass_1', 
    'car_out_overlap', 'car_0', 'car_1', 'car_2', 
    'motor_out_overlap', 'motor_0', 'motor_1'
]

def calc_metrics_from_matrix(mtrx):
    TP = mtrx[1, 1]
    FP = mtrx[0, 1]
    FN = mtrx[1, 0]
    TN = mtrx[0, 0]
    
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return f1 * 100, recall * 100

table_data = []

for algo in algorithms:
    for ml_model in ml_models:
        file_path = f"../CompareOther/{ml_model}/{algo}_{ml_model}.pkl"
        row = {
            "Method": algo_display_names[algo],
            "Model": model_display_names[ml_model],
            "F1": "N/A",
            "Recall": "N/A"
        }
        
        if os.path.exists(file_path):
            data = pd.read_pickle(file_path)

            mtrx = get_score(data['y'], data['decision_scores'])[0]
            f1, recall = calc_metrics_from_matrix(mtrx)
            
            row["F1"] = f"{f1:.2f}%"
            row["Recall"] = f"{recall:.2f}%"
                
        table_data.append(row)

print("\n2. Loading Proposed Method (10 Seeds) using get_score()...")

for ml_model in ml_models:
    f1_across_seeds = []
    recall_across_seeds = []
    
    for seed in range(40, 50):
        tp_mtrx = np.zeros((2, 2))
        
        for subgroup in topological_subgroups:
            file_path = f"../Models/ModelPerformanceSeed/{seed}/{ml_model}/{subgroup}.pkl"
            
            if os.path.exists(file_path):
                data = pd.read_pickle(file_path)
    
                mtrx = get_score(data['y'], data['decision_scores'])[0]
                tp_mtrx += mtrx
        if tp_mtrx.sum() > 0:
            f1, recall = calc_metrics_from_matrix(tp_mtrx)
            f1_across_seeds.append(f1)
            recall_across_seeds.append(recall)

    row = {
        "Method": "Proposed (Mapper)",
        "Model": model_display_names[ml_model],
        "F1": "N/A",
        "Recall": "N/A"
    }
    
    if len(f1_across_seeds) > 0:
        row["F1"] = f"{np.mean(f1_across_seeds):.2f}%"
        row["Recall"] = f"{np.mean(recall_across_seeds):.2f}%"
        
    table_data.append(row)

df_results = pd.DataFrame(table_data)
df_results = df_results.sort_values(by=["Model", "F1"]).reset_index(drop=True)

display(df_results)

In [ ]:
print(df_results)